# ModernBERT Experiment Notebook

This notebook is for running and inspecting the `modernbert_staged` pipeline locally on the MacBook M4 Pro.

Recommended usage:
- run the setup cells first
- execute the `gold` and `end_to_end` cells separately so you can track each stage
- inspect mention failures immediately after `end_to_end`
- only generate a submission after you are happy with the dev behavior


In [ ]:
from __future__ import annotations
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import JSON, Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from pestclef.config import ExperimentConfig
from pestclef.data import load_documents

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")

Project root: /Users/mihai.radulescu/Documents/Master NLP/Sem2/BioNLP/PestCLEF2026/PestCLEF2026
Python: /Users/mihai.radulescu/Documents/Master NLP/Sem2/BioNLP/PestCLEF2026/PestCLEF2026/.venv/bin/python


In [3]:
env_summary = {
    "torch_version": torch.__version__,
    "mps_built": torch.backends.mps.is_built(),
    "mps_available": torch.backends.mps.is_available(),
    "default_device": "mps" if torch.backends.mps.is_available() else "cpu",
}
display(JSON(env_summary))

if torch.backends.mps.is_available():
    x = torch.ones((2, 3), device="mps")
    print("MPS smoke test:", x.device, (x * 2).sum().item())
else:
    print("MPS is not available in this kernel. Runs will fall back to CPU unless you restart into the project venv.")

<IPython.core.display.JSON object>

MPS smoke test: mps:0 12.0


In [4]:
MODEL_NAME = "modernbert_staged"
ENCODER_NAME = "answerdotai/ModernBERT-base"

EPOCHS = 2
BATCH_SIZE = 4
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4

ARTIFACTS_GOLD = PROJECT_ROOT / "artifacts/modernbert_gold_notebook"
ARTIFACTS_E2E = PROJECT_ROOT / "artifacts/modernbert_e2e_notebook"
ARTIFACTS_SUBMIT = PROJECT_ROOT / "artifacts/modernbert_submit_notebook"

DEVICE = "auto"
LOCAL_FILES_ONLY = False

print({
    "model": MODEL_NAME,
    "encoder": ENCODER_NAME,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "device": DEVICE,
    "local_files_only": LOCAL_FILES_ONLY,
})

{'model': 'modernbert_staged', 'encoder': 'answerdotai/ModernBERT-base', 'epochs': 2, 'batch_size': 4, 'train_batch_size': 4, 'eval_batch_size': 4, 'device': 'auto', 'local_files_only': False}


In [5]:
def run_stream(command: list[str], cwd: Path = PROJECT_ROOT) -> int:
    print("$", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    print(f"\n[exit code: {return_code}]")
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}")
    return return_code


def read_json(path: Path):
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


def show_metrics(path: Path) -> None:
    metrics = read_json(path)
    display(JSON(metrics))


def preview_file_tree(root: Path) -> None:
    if not root.exists():
        print(f"Missing: {root}")
        return
    for path in sorted(root.rglob("*")):
        if path.is_file():
            print(path.relative_to(root))


base_config = ExperimentConfig(project_root=PROJECT_ROOT)
train_docs = load_documents("train", base_config)
dev_docs = load_documents("dev", base_config)
test_docs = load_documents("test", base_config)

print({
    "train_docs": len(train_docs),
    "dev_docs": len(dev_docs),
    "test_docs": len(test_docs),
})

FileNotFoundError: [Errno 2] No such file or directory: 'data/json/PestCLEF-2026_train.json'

## Gold-Entity Run

This is the cleanest way to tell whether the relation model itself is improving.

In [ ]:
run_stream([
    sys.executable,
    "scripts/run_baseline.py",
    "--model", MODEL_NAME,
    "--mode", "gold",
    "--artifacts-dir", str(ARTIFACTS_GOLD),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
])

In [ ]:
show_metrics(ARTIFACTS_GOLD / "dev_gold_entity_metrics.json")
preview_file_tree(ARTIFACTS_GOLD)

## End-to-End Run

This trains mention detection plus relation extraction. If this collapses while `gold` looks good, mention extraction is usually the bottleneck.

In [ ]:
run_stream([
    sys.executable,
    "scripts/run_baseline.py",
    "--model", MODEL_NAME,
    "--mode", "end_to_end",
    "--artifacts-dir", str(ARTIFACTS_E2E),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
])

In [1]:
show_metrics(ARTIFACTS_E2E / "dev_end_to_end_metrics.json")
preview_file_tree(ARTIFACTS_E2E)

NameError: name 'show_metrics' is not defined

In [ ]:
error_report = read_json(ARTIFACTS_E2E / "dev_end_to_end_error_report.json")
predicted_entities = read_json(ARTIFACTS_E2E / "dev_predicted_entities.json")["predicted_entities"]

display(Markdown("### Relation Error Summary"))
display(JSON({
    "relation_false_positives": error_report.get("relation_false_positives", {}),
    "relation_false_negatives": error_report.get("relation_false_negatives", {}),
} ))

spurious = error_report.get("entity_detection_errors", {}).get("spurious_entities", [])
print(f"Spurious entities: {len(spurious)}")
pd.DataFrame(spurious[:40])

In [ ]:
DOC_ID = "103963"

doc_lookup = {document.doc_id: document for document in dev_docs}
document = doc_lookup[DOC_ID]
predicted = predicted_entities.get(DOC_ID, [])

print("Document:", DOC_ID)
print("Gold canonical entities:", len(document.canonical_entities))
print("Predicted canonical entities:", len(predicted))

pd.DataFrame(predicted)

In [ ]:
TEXT_PREVIEW_CHARS = 2500
print(document.text[:TEXT_PREVIEW_CHARS])

## Submission Run

This retrains on `train+dev` and writes `submission.csv`.

In [ ]:
run_stream([
    sys.executable,
    "scripts/generate_submission.py",
    "--model", MODEL_NAME,
    "--train-on-dev",
    "--artifacts-dir", str(ARTIFACTS_SUBMIT),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
])

In [ ]:
preview_file_tree(ARTIFACTS_SUBMIT)
submission_df = pd.read_csv(PROJECT_ROOT / "submission.csv")
submission_df.head()